## Streamlined Building Unit Cleaning Code, for San Diego County

Used for joining with SDGE utility

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [2]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [3]:
# load parcels only for San Diego county (will take about 5 minutes)
parcels = gpd.read_file(
   "/../../capstone/electrigrid/data/parcels/Parcels_CA_2014.gdb",
    layer="CA_PARCELS_STATEWIDE",
    where="County='San Diego'").to_crs(epsg=4326)

In [4]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint

Access: https://sat-io.earthengine.app/view/gba

In [5]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

## Streamlined Code 

In [6]:
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    predicate = "intersects")

building_zillow = building_zillow.drop(columns = "index_right")

# add in the parcel number to add all of the building volumes for the regression
building_zillow_parcels = building_zillow.sjoin(
                                parcels,
                                how = "left",
                                predicate = "intersects")

# drop the unnecessary columns
building_zillow_parcels = building_zillow_parcels.drop(columns = ['index_right', 'ADDRESS', 'CITY', 'ZIP','Shape_Length', 'Shape_Area'])

# reproject data frame to crs with meters as units
building_m = building_zillow_parcels.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

# aggregate the volumes for the unit regression by parcel
total_volume_m3 = building_w_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'PARNO')

# some of the homes don't have a parcel number so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

# run model
results = smf.ols('unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'PARNO')

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace anywhere the missing_total_volume is missing
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop residual column
multi_complete = multi_complete.drop(['residual'], axis = 1)

# drop the parno column
multi_complete = multi_complete.drop(columns = 'PARNO')

# drop the unnecessary columns of the parcels_res data
parcels_res = parcels_res.drop(columns = ['ADDRESS', 'CITY', 'ZIP', 'unit'])

multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")

# remove duplicates so that we don't end up with multiple centroids per parcel
multi_by_parcel = multi_by_parcel.drop_duplicates(subset = 'PARNO', keep = 'first')

# change crs so centroids are made accurately
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")


## IN ORDER TO CONCAT THE MULTI-FAMILY HOMES WITH SINGLE FAMILY HOMES THEY MUST ALL BE POINTS
# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# rename the centroid to geometry
multi_by_parcel = multi_by_parcel.rename(columns = {'centroids' : 'geometry'})

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('geometry')

# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

# subset for single family zillow and condos
non_multi_zillow = zillow[zillow['type'] != "Multi"].to_crs(zillow.crs)

# concat the single and multifamily dataframes
complete_zillow = pd.concat([multi_by_parcel_processed, non_multi_zillow], axis = 0)

/tmp/ipykernel_1957386/1141808750.py:94: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_1957386/1141808750.py:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [7]:
complete_zillow

,PARNO,County_left,Shape_Length,Shape_Area,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,County_right,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,geometry
151,4236921300,San Diego,69.352091,238.641330,7349122.0,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,San Diego,140.235708,431.200368,431.200368,NaN,POINT (-117.25142 32.76728)
154,4236220200,San Diego,67.240870,224.046738,7349569.0,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,San Diego,117.733224,320.096607,320.096607,NaN,POINT (-117.25230 32.77765)
159,4245420100,San Diego,94.646744,497.462992,7316916.0,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,San Diego,217.808548,1689.639778,3445.225886,NaN,POINT (-117.23320 32.79144)
160,4245420700,San Diego,92.385658,398.294157,7316886.0,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,2.0,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,San Diego,113.884017,734.439912,734.439912,NaN,POINT (-117.23297 32.79055)
192,5670903600,San Diego,298.920714,4397.340123,7106151.0,osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,40.0,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,San Diego,1895.486015,12525.523059,12525.523059,NaN,POINT (-117.09531 32.63634)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10012615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,2.0,None,None,O,NaN,1737925.0,living,2408.0,10714295,06111002500,h2540,Others,RR101,NaN,NaN,NaN,NaN,NaN,POINT (-119.26588 34.25417)
10012616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,NaN,921196.0,living,3301.0,10714296,06111002500,h2540,Others,RR101,NaN,NaN,NaN,NaN,NaN,POINT (-119.26591 34.25429)
10012617,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,NaN,1503087.0,living,2667.0,10714297,06111002500,h2540,Others,RR101,NaN,NaN,NaN,NaN,NaN,POINT (-119.26594 34.25441)
10012618,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,NaN,998162.0,living,3011.0,10714430,06111002500,h2540,Others,RR101,NaN,NaN,NaN,NaN,NaN,POINT (-119.26252 34.25562)


## Final results

In [38]:
multi_complete.head() # should be my buildings??

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-116.99693 33.29913, -116.99687 33.2...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,412.421351,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-116.98655 33.25268, -116.98620 33.2...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,452.260448,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,232.343850,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-116.37224 33.25284, -116.37224 33.2...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,284.844083,1177.213436
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-116.37297 33.25286, -116.37283 33.2...",Multi,1962.0,4.0,None,None,None,2.0,155077.0,living,1736.0,7599626,06073021002,h479,SDGE,RI101,342.459460,1328.097228


In [8]:
multi_summed_units.head(3)

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit,unit_right
23,4241720700,San Diego,None,None,None,98.936758,434.984710,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,...",3.0,NaN
118,4241721300,San Diego,None,None,None,76.116160,347.815384,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,...",2.0,NaN
119,4241722000,San Diego,None,None,None,106.557535,579.922629,"MULTIPOLYGON Z (((-117.23549 32.79698 0.00000,...",3.0,NaN


In [9]:
multi_by_parcel.head(3)

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit_left,index_right,...,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,80603,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,130.144663,498.084000
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,7349122,...,1026707.0,None,NaN,7996232.0,06073007602,h567,SDGE,RI101,140.235708,431.200368
154,4236220200,San Diego,None,None,None,67.240870,224.046738,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,...",3.0,80751,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.155171,171.388014


In [10]:
non_multi_points.head(3)

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,...,geometry,index_right,source,id,height_m,var,region,bbox,area_m2,volume_m3
7160203,Single,1959.0,3.0,None,None,None,1.0,179462.0,living,1780.0,...,POINT (-117.24993 33.38692),6701574,ms,UnitedStates_023013203_33032,3.835961,0.299546,USA,"{'xmin': -117.25014289023119, 'ymin': 33.38687...",301.592312,1156.896305
7160213,Single,1952.0,2.0,None,None,I,1.0,153443.0,living,1510.0,...,POINT (-117.24838 33.38564),6698354,ms,UnitedStates_023013203_96432,5.221156,0.925386,USA,"{'xmin': -117.24844943872387, 'ymin': 33.38562...",192.020400,1002.568394
7160303,Single,1962.0,3.0,None,None,I,2.0,401364.0,living,1540.0,...,POINT (-117.24780 33.38609),6698329,ms,UnitedStates_023013203_294280,6.511510,1.177574,USA,"{'xmin': -117.24791472273105, 'ymin': 33.38604...",237.514781,1546.579963


## Renaming and saving

In [13]:
multi_summed_units_sd = multi_summed_units.copy()

multi_by_parcel_sd = multi_by_parcel.copy()

non_multi_points_sd = non_multi_points.copy()

In [17]:
# takes a really long time :,(
multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 

In [15]:
non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

In [16]:
multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

In [14]:
## Saving! (takes at least 40 minutes)

multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 